# PSG Asymmetry Pipeline
This notebook generates the Asym_* spectrum folders by interpolating between the reference mid-transit configuration and the extreme ingress/egress models, then downloads the spectra from NASA PSG.

In [5]:
import numpy as np
import os
import re
import requests

In [6]:
PSG_URL = "https://psg.gsfc.nasa.gov/api.php"
cfg_templates = {
    "ingress": "Asym_1.0/wasp17b_transit_ingress_ultra_hazy_cold.cfg",
    "mid": "Asym_1.0/wasp17b_transit_mid_ultra_reference.cfg",
    "egress": "Asym_1.0/wasp17b_transit_egress_ultra_clear_hot.cfg"
}
PARAMS = ["ATMOSPHERE-TEMPERATURE", "ATMOSPHERE-CLOUDFRACTION", "ATMOSPHERE-AABUN"]
# PSG CALL (HTML RESPONSE)
def run_psg(cfg_text):
    response = requests.post(
        PSG_URL,
        data=cfg_text.encode("utf-8"),
        headers={"User-Agent": "Mozilla/5.0", "Content-Type": "text/plain"}, timeout=300)
    text = response.text
    if response.status_code != 200:
        raise RuntimeError(f"PSG error: {response.status_code}")
    if "# Wave/freq" not in text:
        raise RuntimeError("PSG returned response without spectrum")
    return text
# EXTRACT SPECTRUM FROM HTML
def extract_spectrum(psg_html):
    start_patterns = ["# Wave/freq", "#Wave/freq"]
    lines = psg_html.splitlines()
    start_idx = -1
    for i, line in enumerate(lines):
        if any(p in line for p in start_patterns):
            start_idx = i
            break
    if start_idx == -1:
        raise RuntimeError("Spectrum header not found")
    data = []
    for l in lines[start_idx:]:
        l = l.strip()
        if not l:
            continue
        if l.startswith("<"):
            continue
        if l.startswith("#") and not l[0].isdigit():
            continue
        parts = l.split()
        if len(parts) < 5:
            continue
        try:
            float(parts[0])
        except:
            continue
        data.append(l)
    if "Wave/freq" not in psg_html:
        raise RuntimeError("No spectrum block in PSG output")
    from io import StringIO
    return np.loadtxt(StringIO("\n".join(data)))
def run_psg_debug(cfg_text, phase="unknown", asym="unknown"):
    response = requests.post(PSG_URL, data=cfg_text.encode("utf-8"), headers={"User-Agent": "Mozilla/5.0", "Content-Type": "text/plain"}, timeout=300)
    print(f"\nPSG DEBUG | phase={phase} | asym={asym}")
    print("STATUS:", response.status_code)
    print("CONTENT LENGTH:", len(response.text))
    text = response.text
    print("\n--- RAW FIRST 80 LINES ---\n")
    for i, line in enumerate(text.splitlines()[:80]):
        print(f"{i:03d}: {line}")
    if "# Wave/freq" in text or "#Wave/freq" in text:
        print("\n[SUCCESS] Spectrum found")
    else:
        print("\n[WARNING] No spectrum header found")
    return text
# CFG HELPERS
def get_value(cfg, tag):
    m = re.search(rf"<{tag}>([-+0-9.eE]+)", cfg)
    return float(m.group(1)) if m else None
def set_value(cfg, tag, val):
    return re.sub(rf"<{tag}>[-+0-9.eE]+", f"<{tag}>{val}", cfg)
# INTERPOLATION (mid ↔ extreme)
def interpolate(mid_cfg, ext_cfg, a):
    out = mid_cfg
    for p in PARAMS:
        mid_v = get_value(mid_cfg, p)
        ext_v = get_value(ext_cfg, p)
        if mid_v is None or ext_v is None:
            continue
        new_v = mid_v + a * (ext_v - mid_v)
        out = set_value(out, p, new_v)
    return out
# MAIN PIPELINE
def run_asymmetry_pipeline():
    # load CFGs
    with open(cfg_templates["ingress"], "r", encoding="utf-8") as f:
        cfg_ing_base = f.read()
    with open(cfg_templates["mid"], "r", encoding="utf-8") as f:
        cfg_mid = f.read()
    with open(cfg_templates["egress"], "r", encoding="utf-8") as f:
        cfg_eg_base = f.read()
    asymmetry_levels = np.arange(0.1, 1.0, 0.1)
    for a in asymmetry_levels:
        folder = f"Asym_{a:.1f}"
        os.makedirs(folder, exist_ok=True)
        print("\nASYMMETRY LEVEL:", a)
        cfg_ing = interpolate(cfg_mid, cfg_ing_base, a)
        cfg_eg  = interpolate(cfg_mid, cfg_eg_base, a)
        scenarios = {"ingress": cfg_ing, "mid": cfg_mid, "egress": cfg_eg}
        for phase, cfg in scenarios.items():
            print("Running:", phase)
            # save cfg
            cfg_path = os.path.join(folder, f"{phase}.cfg")
            with open(cfg_path, "w", encoding="utf-8") as f:
                f.write(cfg)
            # PSG CALL (production)
            html = run_psg(cfg)
            # extract spectrum
            data = extract_spectrum(html)
            # save numeric output
            out_path = os.path.join(folder, f"{phase}.txt")
            np.savetxt(out_path, data)
            print("Saved:", out_path)
    print("\nPIPELINE COMPLETE")

In [10]:
#RUN
run_asymmetry_pipeline()


ASYMMETRY LEVEL: 0.1
Running: ingress
Saved: Asym_0.1\ingress.txt
Running: mid
Saved: Asym_0.1\mid.txt
Running: egress
Saved: Asym_0.1\egress.txt

ASYMMETRY LEVEL: 0.2
Running: ingress
Saved: Asym_0.2\ingress.txt
Running: mid
Saved: Asym_0.2\mid.txt
Running: egress
Saved: Asym_0.2\egress.txt

ASYMMETRY LEVEL: 0.30000000000000004
Running: ingress
Saved: Asym_0.3\ingress.txt
Running: mid
Saved: Asym_0.3\mid.txt
Running: egress
Saved: Asym_0.3\egress.txt

ASYMMETRY LEVEL: 0.4
Running: ingress
Saved: Asym_0.4\ingress.txt
Running: mid
Saved: Asym_0.4\mid.txt
Running: egress
Saved: Asym_0.4\egress.txt

ASYMMETRY LEVEL: 0.5
Running: ingress
Saved: Asym_0.5\ingress.txt
Running: mid
Saved: Asym_0.5\mid.txt
Running: egress
Saved: Asym_0.5\egress.txt

ASYMMETRY LEVEL: 0.6
Running: ingress
Saved: Asym_0.6\ingress.txt
Running: mid
Saved: Asym_0.6\mid.txt
Running: egress
Saved: Asym_0.6\egress.txt

ASYMMETRY LEVEL: 0.7000000000000001
Running: ingress
Saved: Asym_0.7\ingress.txt
Running: mid
Saved: 